# Batch PNG Cell Analyzer

Select multiple PNG images, tune cell segmentation with a complete preview, and export per-image and combined measurements. PNG input is treated as one fixed intensity channel, so no channel selection is needed.

In [1]:
from copy import deepcopy
from pathlib import Path

import pandas as pd
from IPython.display import display

from cell_analyzer.config import load_yaml
from cell_analyzer.interactive import launch_batch_tuning_widget

## 1. Set batch paths and PNG calibration

`PNG_PATHS` may be left empty because the interface provides a native multi-file picker. Enter the total physical width and height of the entire PNG in micrometers. The program divides these values by each PNG's pixel width and height to calculate the X/Y `µm/pixel` calibration. All PNG files in one batch are assumed to cover the same total physical width and height.

In [2]:
CONFIG_PATH = Path(r"E:\projects\cell_analyzer\notebook_config.yaml")
OUTPUT_ROOT = Path(r"E:\projects\cell_analyzer\output\batch_png_results")
PNG_PATHS = []  # Optional: add absolute PNG paths here, or use the interface.

# Required for calibrated µm² output. Replace None with the total physical
# width and height of the entire PNG, not the size of one pixel.
IMAGE_WIDTH_UM = 1200   # Example: 500.0
IMAGE_HEIGHT_UM = 500  # Example: 250.0

config = load_yaml(CONFIG_PATH)
config_input = config.setdefault("input", {})
config_input.pop("czi_path", None)
config_input.pop("image_path", None)
config_input.pop("pixel_size_um_x", None)
config_input.pop("pixel_size_um_y", None)
config_input.update(
    {
        "scene": 0,
        "time_index": 0,
        "z_projection": "max",
        "z_index": 0,
        "zoom": 1.0,
        "segmentation_channel": 0,
        "image_width_um": IMAGE_WIDTH_UM,
        "image_height_um": IMAGE_HEIGHT_UM,
    }
)

# A PNG is analyzed as one fixed intensity channel. Reuse channel-0 processing
# defaults from the YAML and discard the unused CZI channel definitions.
intensity_config = deepcopy(config.get("channels", {}).get("0", {}))
intensity_config["alias"] = "Intensity"
config["channels"] = {"0": intensity_config}
config

{'input': {'output_dir': 'E:\\projects\\cell_analyzer\\output\\notebook_run',
  'scene': 0,
  'time_index': 0,
  'z_projection': 'max',
  'z_index': 0,
  'zoom': 1.0,
  'segmentation_channel': 0,
  'image_width_um': 1200,
  'image_height_um': 500},
 'segmentation': {'threshold_method': 'otsu',
  'threshold_scale': 0.8,
  'threshold_percentile': 90.0,
  'adaptive_block_size_px': 51,
  'adaptive_offset': 0.0,
  'invert': False,
  'opening_radius_px': 1,
  'closing_radius_px': 2,
  'fill_all_holes': True,
  'min_hole_area_px': 32,
  'min_area_um2': 5.0,
  'max_area_um2': None,
  'min_circularity': 0.05,
  'min_local_contrast_ratio': 1.0,
  'local_contrast_ring_px': 4,
  'clear_border': True,
  'border_exclusion_margin_px': 10,
  'split_touching': False,
  'min_peak_distance_px': 8,
  'watershed_min_peak_height_px': 0.0,
  'watershed_min_peak_prominence_px': 0.0,
  'watershed_compactness': 0.0},
 'channels': {'0': {'alias': 'Intensity',
   'gaussian_sigma_px': 1.0,
   'measurement_threshol

## 2. Launch the multi-file PNG interface

First fill in `IMAGE_WIDTH_UM` and `IMAGE_HEIGHT_UM` above. Then use **Add images** to choose multiple PNG files. Pick an image under **Preview file**, then click **Load preview**. PNG input has one fixed intensity channel, so the parameter-channel and segmentation-channel selectors are hidden. Each image remembers its own segmentation settings; use **Apply current settings to all** when the entire batch should share them, then click **Run all files**.

In [3]:
batch_tuner = launch_batch_tuning_widget(
    config,
    image_paths=PNG_PATHS or None,
    output_root=OUTPUT_ROOT,
    max_preview_dimension=1400,
)
batch_tuner

## 3. Inspect batch results

Run this cell only after the interface reports that the PNG batch is complete.

In [6]:
batch_result = batch_tuner.batch_result
if batch_result is None:
    print("Click 'Run all files' and wait for completion first.")
else:
    batch_summary = pd.read_csv(batch_result["files"]["batch_summary_csv"])
    combined_measurements = pd.read_csv(
        batch_result["files"]["combined_measurements_csv"]
    )
    print(
        f"Completed: {batch_result['completed']} | Failed: {batch_result['failed']} | "
        f"Total ROIs: {batch_result['total_roi_count']}"
    )
    display(batch_summary)
    display(combined_measurements)

Completed: 38 | Failed: 0 | Total ROIs: 413


,batch_file_index,source_file,source_name,status,roi_count,output_dir,error
0,1,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#1 cortex 1200x500.png,complete,15,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,NaN
1,2,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#1 hippo 1200x500.png,complete,5,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,NaN
2,3,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#2 cortex 1200x500.png,complete,17,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,NaN
3,4,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#2 hippo 1200x500.png,complete,15,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,NaN
4,5,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#3 cortex 1200x500.png,complete,26,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,NaN
5,6,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#3 hippo 1200x500.png,complete,8,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,NaN
6,7,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD1 TS cortex 1200x500.czi #2.png,complete,19,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,NaN
7,8,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD1 TS hippo 1200x500.czi #2.png,complete,32,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,NaN
8,9,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD2 TS cortex 1200x500.png,complete,10,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,NaN
9,10,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD2 TS hippo 1200x500.png,complete,9,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,NaN


,batch_file_index,source_file,source_name,roi_id,roi_area_analysis_px,roi_area_source_px,roi_area_um2,perimeter_analysis_px,circularity,intensity_mean_intensity,intensity_integrated_intensity,intensity_positive_area_um2,intensity_positive_fraction
0,1,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#1 cortex 1200x500.png,1,92.0,92.0,35.773352,34.142136,0.991782,145.037900,13343.486822,35.773352,1.0
1,1,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#1 cortex 1200x500.png,2,441.0,441.0,171.478787,99.053824,0.564815,110.262435,48625.733669,171.478787,1.0
2,1,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#1 cortex 1200x500.png,3,1065.0,1065.0,414.115439,172.551299,0.449493,123.311228,131326.457504,414.115439,1.0
3,1,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#1 cortex 1200x500.png,4,386.0,386.0,150.092544,87.982756,0.626617,120.103576,46359.980246,150.092544,1.0
4,1,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,dorsal AD#1 cortex 1200x500.png,5,207.0,207.0,80.490043,55.112698,0.856400,108.192141,22395.773258,80.490043,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,38,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,ventral PAC-cKO-AD2 TS hippo 1200x500.png,4,59.0,59.0,91.766426,25.899495,1.105298,85.642171,5052.888073,91.766426,1.0
409,38,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,ventral PAC-cKO-AD2 TS hippo 1200x500.png,5,28.0,28.0,43.550168,17.313708,1.173783,84.061543,2353.723194,43.550168,1.0
410,38,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,ventral PAC-cKO-AD2 TS hippo 1200x500.png,6,209.0,209.0,325.070899,55.112698,0.864675,116.782922,24407.630760,325.070899,1.0
411,38,I:\QIULAB\data\AD_PAC\AD-PAC_TSstain\20260130 ...,ventral PAC-cKO-AD2 TS hippo 1200x500.png,7,190.0,190.0,295.518999,50.970563,0.919019,129.895380,24680.122204,295.518999,1.0
